# Memory: what the model knows between turns

A model only knows the messages included in the current call. This notebook shows three ways an application can carry the right context forward: manual history, manual summarisation, and a checkpointer.


In [ ]:
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage, HumanMessage, AIMessage

model = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq",
    reasoning_effort="low",
)


## 1. Manual memory

The list below is the memory. The model knows Ada's name on turn two only because the application sends the complete message list again.


In [ ]:
messages = [SystemMessage("You are a concise assistant.")]

# Turn 1
messages.append(HumanMessage("My name is Ada. I am learning about LLMs."))
reply = model.invoke(messages)
messages.append(AIMessage(reply.content))
print("AI:", reply.content)

# Turn 2: the model only knows Ada because we resend the whole list.
messages.append(HumanMessage("What is my name and what am I learning?"))
reply = model.invoke(messages)
messages.append(AIMessage(reply.content))
print("AI:", reply.content)

print("Messages stored:", len(messages))


## 2. Manual memory plus summarisation

History costs tokens. We can ask the model to summarise old messages, then keep the system instruction and the summary instead of every detailed turn.


In [ ]:
# Pretend a long conversation already happened.
messages = [
    SystemMessage("You are a concise assistant."),
    HumanMessage("My name is Ada. I am learning about LLMs."),
    AIMessage("Nice to meet you, Ada!"),
    HumanMessage("I like Python and I use Groq."),
    AIMessage("Great combo."),
]

# Step 1: summarise every old message except the system instruction.
old_text = "\n".join(f"{message.type}: {message.content}" for message in messages[1:])
summary = model.invoke(
    f"Summarize this conversation in one line:\n{old_text}"
).content
print("Summary:", summary)

# Step 2: replace detailed history with a compact summary.
messages = [
    messages[0],
    SystemMessage(f"Summary of earlier chat: {summary}"),
]
print("Messages stored after summary:", len(messages))

# Step 3: the model keeps useful context using fewer input tokens.
messages.append(HumanMessage("What is my name and what do I use?"))
print("AI:", model.invoke(messages).content)


## 3. Checkpointer

A checkpointer stores the chat history for a named thread. The same `thread_id` restores the same conversation. A different `thread_id` starts with an empty history.


In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# The checkpointer is the notebook that stores chat history.
memory = InMemorySaver()

agent = create_agent(
    model,
    tools=[],
    system_prompt="You are a concise assistant.",
    checkpointer=memory,
)

# The same thread ID means the same stored conversation.
user1_config = {"configurable": {"thread_id": "user1"}}

# Turn 1: send only the new message.
message1 = HumanMessage("My name is Ada. I am learning about LLMs.")
response1 = agent.invoke({"messages": [message1]}, user1_config)
print("AI:", response1["messages"][-1].content)

# Turn 2: the checkpointer restores user1's history.
message2 = HumanMessage("What is my name and what am I learning?")
response2 = agent.invoke({"messages": [message2]}, user1_config)
print("AI:", response2["messages"][-1].content)

# A different thread ID starts with empty memory.
user2_config = {"configurable": {"thread_id": "user2"}}
message3 = HumanMessage("What is my name?")
response3 = agent.invoke({"messages": [message3]}, user2_config)
print("AI:", response3["messages"][-1].content)
